# `mutate` — Reference

`mutate` creates or overwrites columns using clean keyword argument syntax (`col="expr"` or `col=callable`). Each entry is evaluated in order via pandas `eval()` — a plain formula per column, or a lambda when needed.

### Calling Styles

1. **Keyword arguments** (most Pythonic): `.pt.mutate(bmi="body_mass_g / bill_length_mm ** 2", mass_kg="body_mass_g / 1000")`
2. **External file specification**: `.pt.mutate("@features.txt")`

---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [2]:
# Body mass index style ratio — clean assignment formula via kwargs
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,body_mass_g,bill_length_mm,bmi
327,Gentoo,5500.0,53.4,1.928769
83,Adelie,4200.0,35.1,3.409063
39,Adelie,4650.0,39.8,2.935532
72,Adelie,3550.0,39.6,2.263800
257,Gentoo,5250.0,44.4,2.663136
2,Adelie,3250.0,40.3,2.001121
300,Gentoo,4625.0,49.1,1.918442
118,Adelie,3350.0,35.7,2.628502
242,Gentoo,4400.0,46.5,2.034917
140,Adelie,3400.0,40.2,2.103908


In [3]:
# Column names inside the expression must stay unquoted — quoting one turns it into
# a string literal, which will break arithmetic
try:
    penguins.pt.mutate(bmi="'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as exc:
    print(f"TypeError (as expected): {exc}")


TypeError (as expected): unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call via kwargs
(
    penguins
    .pt.mutate(heavy='body_mass_g > 4000', mass_kg='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'heavy')
    .sample(10)
)


,species,body_mass_g,mass_kg,heavy
147,Adelie,3475.0,3.475,False
231,Gentoo,5550.0,5.550,True
67,Adelie,4100.0,4.100,True
195,Chinstrap,3500.0,3.500,False
106,Adelie,3750.0,3.750,False
122,Adelie,3450.0,3.450,False
238,Gentoo,4800.0,4.800,True
172,Chinstrap,3600.0,3.600,False
287,Gentoo,5800.0,5.800,True
311,Gentoo,5400.0,5.400,True


In [5]:
# mass_lb references mass_kg, derived by the keyword just before it
(
    penguins
    .pt.mutate(mass_kg='body_mass_g / 1000', mass_lb='mass_kg * 2.20462')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'mass_lb')
    .sample(10)
)


,species,body_mass_g,mass_kg,mass_lb
72,Adelie,3550.0,3.550,7.826401
165,Chinstrap,4050.0,4.050,8.928711
120,Adelie,3150.0,3.150,6.944553
281,Gentoo,5300.0,5.300,11.684486
311,Gentoo,5400.0,5.400,11.904948
35,Adelie,4150.0,4.150,9.149173
6,Adelie,3625.0,3.625,7.991747
47,Adelie,2975.0,2.975,6.558744
333,Gentoo,5500.0,5.500,12.125410
137,Adelie,3975.0,3.975,8.763364


## String comparisons and local variables
String literals *inside* the expression (e.g. `'Adelie'`) need real quotes — column names stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.


In [6]:
# String comparisons inside the expression formula
(
    penguins
    .pt.mutate(is_adelie="species == 'Adelie'")
    .pt.select('species', 'is_adelie')
    .sample(10)
)


,species,is_adelie
88,Adelie,True
115,Adelie,True
122,Adelie,True
79,Adelie,True
309,Gentoo,False
43,Adelie,True
269,Gentoo,False
212,Chinstrap,False
238,Gentoo,False
166,Chinstrap,False


In [7]:
# @-prefixed names resolve against the scope that called mutate(), not the module
# — works identically whether calling pt.mutate() or chaining .pt.mutate()
threshold = 4000

(
    penguins
    .pt.mutate(heavy='body_mass_g >= @threshold')
    .pt.select('species', 'body_mass_g', 'heavy')
    .sample(10)
)


,species,body_mass_g,heavy
42,Adelie,3100.0,False
152,Chinstrap,3500.0,False
291,Gentoo,5000.0,True
86,Adelie,3800.0,False
274,Gentoo,4900.0,True
222,Gentoo,4450.0,True
243,Gentoo,5050.0,True
327,Gentoo,5500.0,True
163,Chinstrap,3775.0,False
30,Adelie,3250.0,False


## Overwriting an existing column

In [8]:
# mutate() can overwrite a column in place, e.g. converting units
(
    penguins
    .pt.mutate(body_mass_g='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g')
    .sample(10)
)


,species,body_mass_g
225,Gentoo,4.55
321,Gentoo,5.60
165,Chinstrap,4.05
213,Chinstrap,3.65
259,Gentoo,5.35
171,Chinstrap,4.40
181,Chinstrap,4.55
84,Adelie,3.35
41,Adelie,3.90
177,Chinstrap,4.15


## Column names with spaces — backtick quoting
`pandas.eval()` uses **backticks**, not the single/double quotes used elsewhere in pytae, to reference a column name containing a space.

In [9]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3750, 4200], 'bill length mm': [39.1, 46.5]})

# Backticks protect column names that contain spaces
spaced.pt.mutate(bmi="`body mass g` / `bill length mm` ** 2")


,body mass g,bill length mm,bmi
0,3750,39.1,2.452888
1,4200,46.5,1.942421


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [10]:
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.qry(bmi='> 2')
    .pt.select('species', 'island', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,island,body_mass_g,bill_length_mm,bmi
279,Gentoo,Biscoe,5550.0,50.4,2.184902
1,Adelie,Torgersen,3800.0,39.5,2.435507
299,Gentoo,Biscoe,5950.0,45.2,2.912327
297,Gentoo,Biscoe,6000.0,51.1,2.297785
109,Adelie,Biscoe,4775.0,43.2,2.558621
342,Gentoo,Biscoe,5200.0,45.2,2.545227
260,Gentoo,Biscoe,3950.0,42.7,2.166413
314,Gentoo,Biscoe,4850.0,44.5,2.449186
47,Adelie,Dream,2975.0,37.5,2.115556
49,Adelie,Dream,4150.0,42.3,2.319356


## Conditional column creation — `if_else()`, `case_when()`, `map()`
Plain `eval()` has no ternary/`where()` support, so `mutate()` provides three dplyr-style helpers as ordinary function calls, evaluated via `np.where()`/`np.select()`/`Series.map()`: `if_else(condition, true_value, false_value)`, `case_when((cond1, val1), (cond2, val2), ..., default)`, and `map(column, {key: value, ...}, default)`. A trailing bare argument to `case_when` is the catch-all default (like SQL ELSE). Because they are ordinary calls, they compose and chain with each other and with any pandas method. String outcomes need quotes; conditions are vectorized — prefer `and`/`or`/`not` (the bitwise `&`/`|`/`~` also work).

In [11]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
# true_value and false_value can be scalars or column names; conditions are vectorized
(
    penguins
    .pt.mutate(weight_class="if_else(body_mass_g > 4000, 'heavy', 'light')")
    .pt.select('species', 'body_mass_g', 'weight_class')
    .sample(10)
)


,species,body_mass_g,weight_class
334,Gentoo,4375.0,heavy
95,Adelie,4300.0,heavy
93,Adelie,4450.0,heavy
42,Adelie,3100.0,light
300,Gentoo,4625.0,heavy
173,Chinstrap,3400.0,light
79,Adelie,4000.0,light
335,Gentoo,5850.0,heavy
156,Chinstrap,3725.0,light
204,Chinstrap,3600.0,light


In [12]:
# case_when supports tuples or flat pairs with default=
(
    penguins
    .pt.mutate(
        size_class="case_when(body_mass_g >= 4500, 'large', body_mass_g >= 3500, 'medium', default='small')"
    )
    .pt.select('species', 'body_mass_g', 'size_class')
    .sample(10)
)


,species,body_mass_g,size_class
316,Gentoo,4925.0,large
221,Gentoo,5700.0,large
291,Gentoo,5000.0,large
136,Adelie,3175.0,small
236,Gentoo,4150.0,medium
333,Gentoo,5500.0,large
83,Adelie,4200.0,medium
266,Gentoo,4200.0,medium
313,Gentoo,5650.0,large
332,Gentoo,4650.0,large


In [13]:
# map(column, {key: value, ...}, default) — recode a column through a dictionary lookup
# unmapped keys become default if given, else NaN
(
    penguins
    .pt.mutate(
        island_code="map(island, {'Torgersen': 'TOR', 'Biscoe': 'BIS'}, 'OTH')"
    )
    .pt.select('species', 'island', 'island_code')
    .sample(10)
)


,species,island,island_code
194,Chinstrap,Dream,OTH
104,Adelie,Biscoe,BIS
161,Chinstrap,Dream,OTH
139,Adelie,Dream,OTH
317,Gentoo,Biscoe,BIS
195,Chinstrap,Dream,OTH
34,Adelie,Dream,OTH
202,Chinstrap,Dream,OTH
233,Gentoo,Biscoe,BIS
295,Gentoo,Biscoe,BIS


## First non-null resolution — `coalesce()`

`coalesce(col1, col2, ..., default)` evaluates candidates left-to-right and returns the first non-null value per row, like SQL `COALESCE()` or `dplyr::coalesce()`:

In [14]:
contacts = pd.DataFrame({
    'mobile': [None, '555-1234', None],
    'home': ['555-5678', None, None],
    'work': [None, None, '555-9012'],
})

contacts.pt.mutate(preferred_contact="coalesce(mobile, home, work, 'N/A')")


,mobile,home,work,preferred_contact
0,NaN,555-5678,NaN,555-5678
1,555-1234,NaN,NaN,555-1234
2,NaN,NaN,555-9012,555-9012


## Combining expressions and callables in kwargs

`mutate()` cleanly mixes expression strings and python callables (e.g. lambdas) in keyword arguments:


In [15]:
(
    penguins
    .pt.mutate(
        bmi='body_mass_g / bill_length_mm ** 2',
        mass_kg='body_mass_g / 1000',
        is_heavy=lambda df: df['body_mass_g'] > 4000,
    )
    .pt.select('species', 'bmi', 'mass_kg', 'is_heavy')
    .head(5)
)


,species,bmi,mass_kg,is_heavy
0,Adelie,2.452888,3.75,False
1,Adelie,2.435507,3.80,False
2,Adelie,2.001121,3.25,False
3,Adelie,NaN,NaN,False
4,Adelie,2.561456,3.45,False


## Loading specs from an external file — `@specs.txt`

For complex feature engineering or shared pipelines across Python and the CLI, specs can be loaded from a file with `#` comments and multiline expressions:

In [16]:
# Write a small demo spec file with comments and assignment '=' syntax
spec_content = """# Engineering features for penguins
mass_kg = body_mass_g / 1000

# Convert kg to lbs
mass_lb = mass_kg * 2.20462
"""
with open("penguin_features.txt", "w") as f:
    f.write(spec_content)

# Load directly using @filename
result = penguins.pt.mutate("@penguin_features.txt")
out = (
    result
    .pt.select("species", "mass_kg", "mass_lb")
    .sample(5)
)

# Clean up demo file
import os
os.remove("penguin_features.txt")

out


,species,mass_kg,mass_lb
64,Adelie,2.85,6.283167
207,Chinstrap,3.45,7.605939
17,Adelie,4.50,9.920790
320,Gentoo,4.85,10.692407
276,Gentoo,4.30,9.479866


## Columns with spaces — SQL-style brackets `[col]`

SQL-style square brackets `[col a]` are supported alongside backticks, avoiding shell backtick substitution hazards:

In [17]:
spaced.pt.mutate(ratio='[body mass g] / [bill length mm]')


,body mass g,bill length mm,ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581
